[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/tareas/T2/tarea.ipynb)

# Tarea 2 · Árboles, ensambles y SVM
**DAMMG · Módulo VI** · trabajo individual, unas 3 h · entrega en Moodle: **domingo 15 de noviembre, 23:59** (tolerancia hasta el miércoles 18, 23:59). La solución se publica el jueves 19.

Esta tarea junta lo de los labs 3 y 4 y lo lleva un paso más allá: el corte de un árbol sobre todas las variables (lo que hace scikit-learn en cada nodo), la lectura de sesgo y varianza en los ensambles, el SVM lineal entrenado con tu propio subgradiente y el efecto de $\textcolor{#f27900}{C}$ y $\textcolor{#f27900}{\gamma}$.

**Cómo se califica**
| Parte | Qué | Peso |
|---|---|---|
| A y C | Funciones `graded`: `gini`, `mejor_corte_multi`, `costo_svm`, `subgradiente_svm` (pruebas automáticas del evaluador) | 60 % |
| B | Pregunta de interpretación 1 (celda `interpretacion`) | 20 % |
| D | Pregunta de interpretación 2 (celda `interpretacion`) | 20 % |

Cada pregunta se califica con una rúbrica: **logrado** (afirmación correcta + números de tu notebook + el porqué), **en desarrollo** (correcta pero sin evidencia o con un hueco), **insuficiente** (diagnóstico equivocado) y **sin respuesta**. Cada pregunta dice al final qué debe tener una respuesta *logrado*.

**Reglas.** No cambies nombres ni argumentos de las funciones `graded`; dentro de ellas usa solo `numpy` y `math`: nada de variables globales, `du.` ni funciones de otras celdas (el evaluador solo ve las celdas `graded`). Escribe tus respuestas en las celdas marcadas **Tu respuesta**, máximo 120 palabras cada una. Puedes consultar lo que quieras; las respuestas son tuyas.

**Fuentes.** Ng, *MLS* C2 S4 (árboles, Gini/entropía, bosques, boosting). Ng, *ML* (2011) semana 7 y CS229, *lecture notes* 3 (SVM, margen suave, kernels). Shalev-Shwartz et al. (2011), *Pegasos* (subgradiente para el SVM primal).

## Preparación

> **Antes de escribir nada**
> 1. *Archivo*, opción *Guardar una copia en Drive* y trabaja en esa copia; si no, al cerrar la pestaña pierdes todo.
> 2. Ejecuta las celdas **en orden**. Si reinicias el entorno o algo falla raro: *Entorno de ejecución*, opción *Reiniciar y ejecutar todo*.
> 3. Escribe dentro de las celdas que ya existen (no copies/pegues celdas): así no pierden la etiqueta que usa el evaluador.
>
> *Palabras en inglés que usamos tal cual:* **baseline** (línea base: el modelo tonto que hay que vencer) · **pipeline** (cadena de preprocesamiento + modelo) · **proxy** (variable sustituta de lo que de verdad queremos medir) · **bootstrap** (remuestreo con reemplazo) · **chat waterfall** (todos escriben su respuesta en el chat y la envían al mismo tiempo) · **precision** y **recall** (los nombres de sklearn para precisión y exhaustividad).

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dammg_utils import COLORES
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import average_precision_score
du.estilo_dammg()
np.set_printoptions(precision=4, suppress=True)

In [ ]:
df = pd.read_parquet(RUTA_DATOS)
tr = df[df["particion"] == "train"].copy()
va = df[df["particion"] == "val"].copy()
UMBRAL = UMBRAL_EVENTO                          # celda L0: percentil 85 de |future_net| en train, congelado (Lab 2)
assert UMBRAL == float(tr["future_net"].abs().quantile(0.85))
y_tr, y_va = ((d["future_net"].abs() >= UMBRAL).astype(int).to_numpy() for d in (tr, va))

HORAS_PICO = [7, 8, 9, 17, 18, 19, 20]


def variables(d):
    # Las variables del Lab 3: rezagos ≥ 24 h, calendario y capacidad
    return pd.DataFrame({
        "net_lag_24h": d["net_lag_24h"], "net_lag_168h": d["net_lag_168h"], "net_mean_24h": d["net_mean_24h"],
        "abs_net_lag_24h": d["net_lag_24h"].abs(), "abs_net_lag_168h": d["net_lag_168h"].abs(),
        "abs_net_mean_24h": d["net_mean_24h"].abs(),
        "departures_mean_24h": d["departures_mean_24h"], "departures_mean_168h": d["departures_mean_168h"],
        "departures_lag_168h": d["departures_lag_168h"], "capacity": d["capacity"],
        "is_weekend": d["is_weekend"], "pico": d["hour_of_day"].isin(HORAS_PICO).astype(int),
        "hour_of_day": d["hour_of_day"], "day_of_week": d["day_of_week"],
    }, index=d.index)


X_tr, X_va = variables(tr), variables(va)
print(f"train {X_tr.shape} · val {X_va.shape} · prevalencia val = {y_va.mean():.3f}")

## Parte A · El corte de un nodo, sobre todas las variables (O4.1)
En el Lab 3 buscaste el mejor umbral de **una** variable con entropía. Un árbol real hace eso para **todas** las variables y se queda con la mejor pareja (variable, umbral). Aquí usamos el criterio por defecto de scikit-learn, el **Gini**: $G = 1-\sum_k p_k^2$, y la ganancia
$$\Delta G = G(\text{padre}) - \tfrac{N_\text{izq}}{m}G(\text{izq}) - \tfrac{N_\text{der}}{m}G(\text{der}),$$ con $N_\text{izq}, N_\text{der}$ los ejemplos de cada hijo (en la raíz, el padre tiene los $m$).

In [ ]:
import numpy as np


def gini(conteos):
    """Impureza de Gini de un nodo: G = 1 − Σ p_k², con p_k = conteo_k / total.

    conteos : lista o arreglo de enteros ≥ 0, uno por clase. Nodo vacío (total 0): regresa 0.0.
    Regresa un float. Un nodo puro vale 0; uno 50/50 con dos clases vale 0.5.
    """
    c = np.asarray(conteos, dtype=float)

    # TODO: proporciones y luego 1 − Σ p². Cuida el nodo vacío.
    raise NotImplementedError("Completa gini (Parte A): busca el # TODO.")

In [ ]:
du.revisa_aislada(gini, graded=['mejor_corte_multi', 'costo_svm', 'subgradiente_svm'])
# Autocomprobación de gini
assert np.isclose(gini([5, 5]), 0.5), f"Un nodo 5/5 vale 0.5; obtuve {gini([5, 5])}"
assert gini([0, 9]) == 0.0 and gini([0, 0]) == 0.0, "Nodo puro y nodo vacío valen 0.0"
assert np.isclose(gini([1, 5]), 1 - (1 / 6) ** 2 - (5 / 6) ** 2), "Revisa 1 − Σ p²"
assert np.isclose(gini([3, 3, 3]), 2 / 3), "Con 3 clases iguales: 1 − 3·(1/3)² = 2/3"
print("✅ gini pasa la autocomprobación")

In [ ]:
def mejor_corte_multi(X, y):
    """Mejor corte de un árbol sobre TODAS las variables, con reducción de Gini.

    X : (m, n) variables numéricas.   y : (m,) etiquetas de clase.
    Para cada columna j, los candidatos son los puntos medios entre valores DISTINTOS consecutivos
    de X[:, j]; el corte u manda a la izquierda los ejemplos con X[:, j] <= u.
    ganancia = gini(padre) − (n_izq/m)·gini(izq) − (n_der/m)·gini(der)   (usa tu función gini).
    Regresa (j, umbral, ganancia) = (int, float, float) del corte con MAYOR ganancia.
    Empates (diferencia < 1e-12): gana la columna con menor índice y, dentro de ella, el umbral más chico.
    Si ninguna columna tiene candidatos (todas constantes) regresa (None, None, 0.0). Si hay candidatos pero
    todos dan ganancia 0, aplica la misma regla de empate (empieza con mejor ganancia = −∞).
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    # TODO: recorre las columnas en orden y, en cada una, los candidatos de menor a mayor.
    #       Para ir rápido: ordena la columna una vez y usa np.cumsum para contar la izquierda.
    #       Un doble for sencillo también pasa las pruebas (con m de unos miles).
    raise NotImplementedError("Completa mejor_corte_multi (Parte A): busca el # TODO.")

In [ ]:
du.revisa_aislada(mejor_corte_multi, graded=['gini', 'costo_svm', 'subgradiente_svm'])
# Autocomprobación de mejor_corte_multi con datos chicos
X_a = np.array([[5.0, 1.0], [3.0, 2.0], [4.0, 3.0], [1.0, 4.0], [2.0, 5.0], [6.0, 6.0]])
y_a = np.array([0, 0, 0, 1, 1, 1])
j, u, g = mejor_corte_multi(X_a, y_a)
assert (j, u) == (1, 3.5) and np.isclose(g, 0.5), f"La columna 1 separa perfecto en 3.5 (ganancia 0.5); obtuve {(j, u, g)}"
assert mejor_corte_multi(np.ones((4, 2)), [0, 1, 0, 1]) == (None, None, 0.0), "Sin candidatos regresa (None, None, 0.0)"
j2, u2, _ = mejor_corte_multi(np.column_stack([X_a[:, 1], X_a[:, 1]]), y_a)
assert (j2, u2) == (0, 3.5), "Empate entre columnas: gana la de menor índice"
# Aquí sí importa ponderar cada hijo por su tamaño (N_izq/m y N_der/m)
X_p = np.column_stack([np.arange(1.0, 9.0), [3, 1, 4, 1, 5, 9, 2, 6]])
y_p = np.array([0, 0, 0, 0, 0, 1, 1, 0])
j3, u3, g3 = mejor_corte_multi(X_p, y_p)
assert (j3, u3) == (0, 5.5) and np.isclose(g3, 5 / 24), (
    f"Esperaba (0, 5.5, 0.2083) y obtuve {(j3, u3, round(g3, 4))}. ¿Ponderas el Gini de cada hijo por N_hijo/m?")
print("✅ mejor_corte_multi pasa la autocomprobación")

**En MiBici.** Tomamos 3 000 estaciones-hora de train y seis variables. Tu función debe elegir la misma variable y el mismo umbral que la raíz de un `DecisionTreeClassifier(max_depth=1)`.

In [ ]:
COLS_A = ["departures_lag_168h", "departures_mean_168h", "net_lag_168h", "capacity", "hour_of_day", "is_weekend"]
muestra_a = tr.sample(3000, random_state=0)
X_a3 = muestra_a[COLS_A].to_numpy(dtype=float)
y_a3 = (muestra_a["future_net"].abs() >= UMBRAL).astype(int).to_numpy()

j_mib, u_mib, g_mib = mejor_corte_multi(X_a3, y_a3)
tocon = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_a3, y_a3)
t = tocon.tree_
g_sk = t.impurity[0] - (t.weighted_n_node_samples[1] * t.impurity[1] + t.weighted_n_node_samples[2] * t.impurity[2]) / t.weighted_n_node_samples[0]
print(f"tú:      {COLS_A[j_mib]} <= {u_mib} · ganancia Gini {g_mib:.5f}")
print(f"sklearn: {COLS_A[t.feature[0]]} <= {t.threshold[0]} · ganancia Gini {g_sk:.5f}")
assert j_mib == t.feature[0] and np.isclose(u_mib, t.threshold[0]) and np.isclose(g_mib, g_sk), "No coincide con la raíz de sklearn"
print("✅ tu corte es la raíz del árbol de scikit-learn")

## Parte B · Sesgo y varianza en los ensambles (pregunta de interpretación 1)
Tres familias sobre las mismas variables y la misma partición (train oct 2025–jun 2026, val jul), medidas con AP:
- un árbol de profundidad 3 y uno sin límite;
- bosques de 1, 10 y 100 árboles **profundos** (sin límite de profundidad);
- boosting con árboles de profundidad 2 y 10, 100 o 500 rondas.

La celda tarda de 1 a 3 min en Colab gratuito: no la interrumpas.

In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

familias = {
    "árbol max_depth=3": DecisionTreeClassifier(max_depth=3, random_state=0),
    "árbol sin límite": DecisionTreeClassifier(random_state=0),
    "bosque, 1 árbol profundo": RandomForestClassifier(n_estimators=1, n_jobs=2, random_state=0),
    "bosque, 10 árboles profundos": RandomForestClassifier(n_estimators=10, n_jobs=2, random_state=0),
    "bosque, 100 árboles profundos": RandomForestClassifier(n_estimators=100, n_jobs=2, random_state=0),
    "boosting prof. 2, 10 rondas": HistGradientBoostingClassifier(max_depth=2, max_iter=10, early_stopping=False, random_state=0),
    "boosting prof. 2, 100 rondas": HistGradientBoostingClassifier(max_depth=2, max_iter=100, early_stopping=False, random_state=0),
    "boosting prof. 2, 500 rondas": HistGradientBoostingClassifier(max_depth=2, max_iter=500, early_stopping=False, random_state=0),
}
filas = []
for nombre, modelo in familias.items():
    t0 = time.perf_counter()
    modelo.fit(X_tr, y_tr)
    ap_tr = average_precision_score(y_tr, modelo.predict_proba(X_tr)[:, 1])
    ap_va = average_precision_score(y_va, modelo.predict_proba(X_va)[:, 1])
    filas.append({"modelo": nombre, "AP train": ap_tr, "AP val": ap_va, "brecha": ap_tr - ap_va,
                  "segundos": time.perf_counter() - t0})
tabla_ens = pd.DataFrame(filas).set_index("modelo")

fig, ax = plt.subplots(figsize=(9, 4))
pos = np.arange(len(tabla_ens))
ax.bar(pos - 0.2, tabla_ens["AP train"], 0.4, color=COLORES["w"], label="AP train")
ax.bar(pos + 0.2, tabla_ens["AP val"], 0.4, color=COLORES["error"], label="AP val")
ax.axhline(y_va.mean(), color=COLORES["tinta"], ls=":", lw=1, label="azar")
ax.set_xticks(pos, tabla_ens.index, rotation=30, ha="right", fontsize=8)
ax.set(ylabel="AP", title="Train contra validación por familia")
ax.legend(fontsize=8)
plt.show()
tabla_ens.round(3)

### Pregunta de interpretación 1 (O4.2)
Con los números de la tabla: (a) ¿qué le pasa a la AP de train y de validación al pasar de 1 a 100 árboles en el bosque? ¿El bosque reduce sesgo o varianza? (b) ¿Y al pasar de 10 a 500 rondas de boosting? (c) Si tu árbol tuviera AP train 0.99 y AP val 0.30, ¿qué ensamble probarías primero y por qué?

*Para logrado:* di qué familia baja la varianza y cuál el sesgo, con la AP de train **y** de validación de cada una, y conecta tu elección de (c) con ese diagnóstico. "Más árboles" no es lo mismo que "árboles más profundos".

**Tu respuesta:** _(doble clic para editar; escribe aquí mismo, no en una celda nueva; máximo 120 palabras; cita al menos tres números de la tabla)_

## Parte C · Tu propio SVM lineal con subgradiente
El objetivo del margen suave es convexo pero tiene "quiebres" donde un margen vale 1 (la hinge no es derivable ahí). En lugar del gradiente usamos un **subgradiente**: en cada paso solo cuentan los ejemplos **activos**, los que están dentro de la calle o mal clasificados ($y\,f(x) < 1$).
$$J(\textcolor{#0847c4}{\vec w},b) = \tfrac12\lVert\textcolor{#0847c4}{\vec w}\rVert^2 + \textcolor{#f27900}{C}\sum_i \max\!\big(0,\ 1-y^{(i)}(\textcolor{#0847c4}{\vec w}\cdot x^{(i)}+b)\big)$$

⚠ **Convención.** Esta es la escala de `SVC` (suma de hinges, sin $\frac1m$). Si la reescribes como promedio, el mismo modelo corresponde a otra $C$: $\tfrac{1}{m}\sum$ con $C' = mC$. `b` no se regulariza.

In [ ]:
def costo_svm(w, b, X, y, C):
    """Objetivo del SVM de margen suave (primal), con la convención de SVC de sklearn:

        ½‖w‖² + C · Σ_i max(0, 1 − y⁽ⁱ⁾ (w·x⁽ⁱ⁾ + b))

    w : (n,)   b : número   X : (m, n)   y : (m,) con valores en {−1, +1}   C : número > 0.
    Ojo: es una SUMA sobre los ejemplos (no un promedio) y b no se regulariza. Regresa un float.
    """
    w = np.asarray(w, dtype=float)
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    # TODO: márgenes y·(Xw + b), hinge con np.maximum y la suma ponderada por C.
    raise NotImplementedError("Completa costo_svm (Parte C): busca el # TODO.")

In [ ]:
du.revisa_aislada(costo_svm, graded=['gini', 'mejor_corte_multi', 'subgradiente_svm'])
# Autocomprobación de costo_svm
X_c = np.array([[2.0, 0.0], [0.0, 1.0], [-1.0, -1.0]])
y_c = np.array([1.0, 1.0, -1.0])
w_c, b_c = np.array([0.5, 0.5]), 0.0
# márgenes 1.0, 0.5, 1.0; hinges 0, 0.5, 0; J = ½·0.5 + C·0.5
assert np.isclose(costo_svm(w_c, b_c, X_c, y_c, 2.0), 0.25 + 1.0), f"Esperaba 1.25 y obtuve {costo_svm(w_c, b_c, X_c, y_c, 2.0)}"
assert np.isclose(costo_svm(np.array([3.0, 3.0]), 0.0, X_c, y_c, 5.0), 9.0), "Si todos los márgenes son ≥ 1 solo queda ½‖w‖²"
# Con b = 0.5: márgenes 1.5, 1.0, 0.5; hinges 0, 0, 0.5; J = ½·0.5 + 2·0.5 = 1.25 (b no entra en ½‖w‖²)
J_b = costo_svm(w_c, 0.5, X_c, y_c, 2.0)
assert np.isclose(J_b, 1.25), f"Con b = 0.5 esperaba 1.25 y obtuve {J_b}. ¿Regularizaste b? Solo w va en ½‖w‖²."
print("✅ costo_svm pasa la autocomprobación")

In [ ]:
def subgradiente_svm(w, b, X, y, C):
    """Un subgradiente de costo_svm respecto a (w, b). Regresa (dw, db): arreglo (n,) y float.

    Solo los ejemplos con margen y·(w·x + b) < 1 (estrictamente) aportan a la hinge:
        dw = w − C · Σ_{activos} y⁽ⁱ⁾ x⁽ⁱ⁾
        db =   − C · Σ_{activos} y⁽ⁱ⁾
    En un ejemplo con margen exactamente 1 la hinge no es derivable; por convención NO aporta.
    """
    w = np.asarray(w, dtype=float)
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    # TODO: máscara de activos (margen < 1) y las dos fórmulas. b no lleva término de regularización.
    raise NotImplementedError("Completa subgradiente_svm (Parte C): busca el # TODO.")

In [ ]:
du.revisa_aislada(subgradiente_svm, graded=['gini', 'mejor_corte_multi', 'costo_svm'])
# Autocomprobación de subgradiente_svm: en un punto sin márgenes exactamente 1 coincide con la derivada numérica
rng = np.random.default_rng(4)
X_g, y_g = rng.normal(size=(20, 3)), rng.choice([-1.0, 1.0], 20)
w_g, b_g, C_g, h = rng.normal(size=3), 0.3, 0.7, 1e-6
dw, db = subgradiente_svm(w_g, b_g, X_g, y_g, C_g)
num = [(costo_svm(w_g + h * e, b_g, X_g, y_g, C_g) - costo_svm(w_g - h * e, b_g, X_g, y_g, C_g)) / (2 * h) for e in np.eye(3)]
num_b = (costo_svm(w_g, b_g + h, X_g, y_g, C_g) - costo_svm(w_g, b_g - h, X_g, y_g, C_g)) / (2 * h)
assert np.shape(dw) == (3,), "dw debe tener forma (n,)"
assert np.allclose(dw, num, atol=1e-5) and np.isclose(db, num_b, atol=1e-5), f"tú ({dw}, {db}) · numérico ({np.array(num)}, {num_b})"
# Convención en el quiebre: margen exactamente 1 no aporta
dw1, db1 = subgradiente_svm(np.array([1.0, 0.0]), 0.0, np.array([[1.0, 0.0]]), np.array([1.0]), 10.0)
assert np.allclose(dw1, [1.0, 0.0]) and db1 == 0.0, "Un ejemplo con margen exactamente 1 no es activo"
print("✅ subgradiente_svm pasa la autocomprobación")

**Entrena con tu código.** Dos nubes que se traslapan, $C=1$. El paso $\alpha_t = 1/(t + Cm)$ se hace más chico con el tiempo (con un subgradiente, un paso fijo nunca se asienta). Guardamos el mejor $J$ visto y lo comparamos con `SVC(kernel="linear")`, que resuelve el mismo problema por el dual.

In [ ]:
rng_s = np.random.default_rng(11)
X_s = np.vstack([rng_s.normal([-1.0, -1.0], 0.9, (40, 2)), rng_s.normal([1.2, 1.0], 0.9, (40, 2))])
y_s = np.r_[-np.ones(40), np.ones(40)]
C_s = 1.0

w_s, b_s = np.zeros(2), 0.0
historial, mejor_J, mejor_wb = [], np.inf, None
for t in range(1, 3001):
    dw, db = subgradiente_svm(w_s, b_s, X_s, y_s, C_s)
    paso = 1.0 / (t + C_s * len(y_s))
    w_s, b_s = w_s - paso * dw, b_s - paso * db
    J = costo_svm(w_s, b_s, X_s, y_s, C_s)
    historial.append(J)
    if J < mejor_J:
        mejor_J, mejor_wb = J, (w_s.copy(), b_s)

svc_lin = SVC(kernel="linear", C=C_s, tol=1e-8).fit(X_s, y_s)
J_svc = costo_svm(svc_lin.coef_[0], svc_lin.intercept_[0], X_s, y_s, C_s)
print(f"tu SVM:  w = {mejor_wb[0].round(3)}, b = {mejor_wb[1]:.3f}, J = {mejor_J:.4f}")
print(f"SVC:     w = {svc_lin.coef_[0].round(3)}, b = {svc_lin.intercept_[0]:.3f}, J = {J_svc:.4f}")
assert mejor_J <= J_svc * 1.01, "Tu J debería quedar a menos de 1 % del de SVC: revisa signos y la máscara de activos"
print("✅ tu subgradiente llega al mismo mínimo que SVC")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(np.array(historial) - J_svc + 1e-9, color=COLORES["w"])
ax.set(xscale="log", yscale="log", xlabel="iteración", ylabel="J − J_SVC", title="Subgradiente: baja, con saltos")
plt.show()

## Parte D · El efecto de $C$ y $\gamma$ (pregunta de interpretación 2)
Dos lunas con ruido (200 puntos de entrenamiento y 200 de validación), SVM con kernel RBF. Barremos $\textcolor{#f27900}{C}\in\{0.1, 1, 10, 100\}$ y $\textcolor{#f27900}{\gamma}\in\{0.1, 1, 10, 100\}$ y contamos vectores de soporte (VS).

In [ ]:
from sklearn.datasets import make_moons

X_m, y_m = make_moons(n_samples=400, noise=0.3, random_state=1)
y_m = 2 * y_m - 1
X_m_tr, X_m_va, y_m_tr, y_m_va = X_m[:200], X_m[200:], y_m[:200], y_m[200:]

filas_d = []
for C in [0.1, 1, 10, 100]:
    for gamma in [0.1, 1, 10, 100]:
        m = SVC(kernel="rbf", C=C, gamma=gamma).fit(X_m_tr, y_m_tr)
        filas_d.append({"C": C, "γ": gamma, "VS": len(m.support_),
                        "acc train": np.mean(m.predict(X_m_tr) == y_m_tr), "acc val": np.mean(m.predict(X_m_va) == y_m_va)})
tabla_cg = pd.DataFrame(filas_d)
print("accuracy en validación (filas: C, columnas: γ)")
display(tabla_cg.pivot(index="C", columns="γ", values="acc val").round(3))
print("accuracy en train")
display(tabla_cg.pivot(index="C", columns="γ", values="acc train").round(3))
print("vectores de soporte (de 200)")
display(tabla_cg.pivot(index="C", columns="γ", values="VS"))

fig, axs = plt.subplots(1, 4, figsize=(15, 3.6))
xx, yy = np.meshgrid(np.linspace(-2, 3, 160), np.linspace(-1.6, 2.1, 160))
for ax, (C, gamma) in zip(axs, [(1, 0.1), (1, 1), (1, 100), (100, 1)]):
    m = SVC(kernel="rbf", C=C, gamma=gamma).fit(X_m_tr, y_m_tr)
    f = m.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, f, levels=[-1e9, 0, 1e9], colors=[COLORES["datos"], "#fde3e4"], alpha=0.6)
    ax.contour(xx, yy, f, levels=[-1, 0, 1], colors=COLORES["tinta"], linestyles=["--", "-", "--"], linewidths=[1, 2, 1])
    ax.scatter(*X_m_tr[y_m_tr < 0].T, s=12, color=COLORES["w"])
    ax.scatter(*X_m_tr[y_m_tr > 0].T, s=12, color=COLORES["error"])
    ax.set(title=f"C = {C}, γ = {gamma} · {len(m.support_)} VS", xticks=[], yticks=[])
plt.show()

### Pregunta de interpretación 2 (O5.1)
Con la tabla y las figuras: (a) con $\gamma = 1$ fijo, ¿qué pasa con los vectores de soporte y con la calle al subir $C$ de 0.1 a 100? Explícalo con la hinge. (b) Con $C = 1$ fijo, ¿qué pasa con train y validación al subir $\gamma$ de 0.1 a 100? ¿Sesgo o varianza en cada extremo? (c) ¿Qué par $(C, \gamma)$ entregarías y con qué evidencia?

*Para logrado:* explica el efecto de $C$ con la hinge (qué le pasa a $\lVert w\rVert$ y a la calle), nombra sesgo y varianza en los extremos de $\gamma$ con números de train y validación, y elige el par con validación, nunca con train ni con test.

**Tu respuesta:** _(doble clic para editar; escribe aquí mismo, no en una celda nueva; máximo 120 palabras; cita los valores que usas)_

---
## Parte E. De la sesión 2: ¿cuántas bicis salen mañana a las 8? (práctica, sin puntos)

Esta hoja no alcanzamos a hacerla en clase. Es práctica de métricas de regresión (MAE, RMSE, R²) con su propia estación:
no cambia la calificación de la tarea, pero conviene hacerla antes de las partes de árboles. Escriban sus respuestas en la misma hoja.

Corran las celdas de abajo hasta que aparezca `✅ entorno listo`.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'seaborn': 'seaborn==0.13.2',
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error
from dammg_utils import COLORES

du.estilo_dammg()

## Paso 1. Los datos (3 min)

Cada fila es una estación en una hora. La columna que queremos predecir es `departures`: cuántas bicis salieron de verdad.
`departures_lag_168h` son las salidas de esa misma estación, a esa misma hora, hace una semana (168 horas).

- **train** (octubre a junio): con esto se ajusta la recta.
- **julio** (validación): con esto medimos y escogemos. Es lo que le enseñamos a operaciones.
- agosto y septiembre (prueba): no se tocan; son el examen final del curso.

In [ ]:
df = pd.read_parquet(RUTA_DATOS)
train = df[df.particion == "train"]   # octubre a junio: para ajustar
julio = df[df.particion == "val"]     # julio: para medir y escoger

# La recta de ayer: salidas de hoy a partir de las de hace una semana, ajustada solo con train
recta = LinearRegression().fit(train[["departures_lag_168h"]], train.departures)
print(f"recta: salidas = {recta.coef_[0]:.2f} × (hace una semana) + {recta.intercept_:.2f}")

Las 40 estaciones del curso. `salidas_por_hora` es el promedio de julio y `horas_en_0` la fracción de horas de julio
en que no salió ninguna bici. Escojan una: una con mucho movimiento, una tranquila, la que les quede cerca.

**Ojo con las estaciones cerradas.** Si su estación tiene muchos días seguidos en 0, puede que haya estado cerrada
(obras, mantenimiento); escojan otra. En esos días cualquier propuesta que diga 0 "acierta" sin mérito
y las métricas salen mejores de lo que son. En el Paso 2 una celda lo revisa por ustedes.

In [ ]:
estaciones = julio.groupby("station_id").agg(
    capacidad=("capacity", "first"),                         # anclajes de la estación
    salidas_por_hora=("departures", "mean"),                  # promedio de julio
    horas_en_0=("departures", lambda s: (s == 0).mean()),     # fracción de horas sin salidas
).round(2)
estaciones.sort_values("salidas_por_hora", ascending=False)

## Paso 2. El modelo (4 min)

**Cambien el 51 por el número de su estación** y corran. Es la única línea que se edita en toda la hoja.

La celda calcula las dos propuestas para cada hora de julio y luego el pronóstico de mañana.
Para el lunes 3 de agosto a las 8:00, "hace una semana" es el lunes 27 de julio a las 8:00, que sí está en julio.
Como mañana es lunes, las 8:00 se miden solo de lunes a viernes: los sábados y domingos a esa hora casi no sale nadie.

In [ ]:
estacion = 51   # <- cambien este número por el de su estación

assert estacion in estaciones.index, (f"La estación {estacion} no está en la lista; "
                                      f"usen una de: {sorted(estaciones.index.tolist())}")
mia = julio[julio.station_id == estacion].copy()
mia["regla"] = mia.departures_lag_168h                               # las mismas que hace una semana
mia["recta"] = recta.predict(mia[["departures_lag_168h"]])            # la recta de ayer
ocho = mia[(mia.hour_of_day == 8) & (mia.is_weekend == 0)]           # las 8:00 de lunes a viernes de julio

# Mañana, lunes 3 de agosto a las 8:00: se calcula con el lunes 27 de julio a las 8:00
base = mia.loc[mia.hour == "2026-07-27 08:00", "departures"].iloc[0]
manana = {"regla": base, "recta": recta.predict(pd.DataFrame({"departures_lag_168h": [base]}))[0]}
print(f"Estación {estacion}. El lunes 27 de julio a las 8:00 salieron {base} bicis.")
print(f"Pronóstico para el lunes 3 de agosto a las 8:00:  regla {manana['regla']:.0f}   recta {manana['recta']:.1f}")

In [ ]:
# ¿Estuvo cerrada? Buscamos la racha más larga de días seguidos de julio sin ninguna salida
por_dia = mia.groupby(mia.hour.dt.date).departures.sum()   # salidas de cada día de julio
racha, mas_larga = 0, 0
for salidas in por_dia:
    racha = racha + 1 if salidas == 0 else 0                 # se reinicia en cuanto sale una bici
    mas_larga = max(mas_larga, racha)
print(f"Estación {estacion}: {mas_larga} días seguidos de julio en 0.")
if mas_larga >= 3:
    print("Aviso: muchos días seguidos en 0: puede que haya estado cerrada. Escojan otra estación y vuelvan a correr.")

In [ ]:
# Las 8:00 de lunes a viernes de julio: lo real contra las dos propuestas
largo = ocho.melt(id_vars="hour", value_vars=["departures", "regla", "recta"], var_name="serie", value_name="bicis")
largo["serie"] = largo.serie.replace({"departures": "real"})
plt.figure(figsize=(10, 4))
ax = sns.lineplot(data=largo, x="hour", y="bicis", hue="serie", marker="o",
                  palette={"real": COLORES["tinta"], "regla": COLORES["w"], "recta": COLORES["yhat"]})
ax.set(xlabel="julio, de lunes a viernes a las 8:00", ylabel="bicis que salieron", title=f"Estación {estacion}")
plt.xticks(rotation=30)
plt.show()

## Paso 3. La medición (6 min)

Tres números por propuesta, medidos en julio. Recuerden cómo se lee cada uno:

- **MAE**: por cuántas bicis fallamos en una hora típica, sin importar el signo. En bicis.
- **RMSE**: también en bicis, pero el cuadrado castiga más los errores grandes. Si sale mucho mayor que el MAE,
  hay pocas horas con errores muy grandes.
- **R²**: cuánto mejor que decir siempre el promedio de julio de la estación. 1 es perfecto, 0 es igual que el promedio,
  negativo es peor que el promedio. No tiene unidades.

Para la pregunta de operaciones agregamos dos columnas solo con las 8:00 de lunes a viernes (mañana es lunes):
el MAE a esa hora, y el error que **no se pasó en 8 de cada 10 días laborables** (si dice 6, en 8 de cada 10
mañanas laborables de julio fallamos por 6 bicis o menos).

In [ ]:
def mide(d, col):
    """Las métricas de una propuesta (col) en un pedazo de julio (d)."""
    return {"MAE": mean_absolute_error(d.departures, d[col]),
            "RMSE": root_mean_squared_error(d.departures, d[col]),
            "R²": r2_score(d.departures, d[col])}


tabla = pd.DataFrame({col: mide(mia, col) for col in ["regla", "recta"]}).T
tabla["MAE a las 8"] = [mean_absolute_error(ocho.departures, ocho[c]) for c in ["regla", "recta"]]
tabla["8 de 10 días laborables, error de hasta"] = [(ocho[c] - ocho.departures).abs().quantile(0.8)
                                                   for c in ["regla", "recta"]]
tabla.round(2)

In [ ]:
# MAE contra RMSE de las dos propuestas, en barras
barras = tabla[["MAE", "RMSE"]].reset_index(names="propuesta").melt(id_vars="propuesta", var_name="métrica",
                                                                     value_name="bicis")
ax = sns.barplot(data=barras, x="métrica", y="bicis", hue="propuesta",
                 palette={"regla": COLORES["w"], "recta": COLORES["yhat"]}, edgecolor=COLORES["tinta"])
ax.set(title=f"Estación {estacion}, julio completo", xlabel="")
plt.show()

**1. En su tabla, ¿cómo se le dice el MAE de la recta a operaciones?**

- a) En una hora típica de julio, la recta falla por esa cantidad de bicis, para arriba o para abajo
- b) La recta acierta ese porcentaje de las horas
- c) Es el error de la peor hora de julio
- d) Es la proporción de la variación que explica la recta

**Su respuesta (doble clic para escribir):** ___

**2. Si en su tabla el RMSE es mucho más grande que el MAE (en la 51, la recta: 9.2 contra 5.7), ¿qué les dice?**

- a) Que el modelo está mal programado
- b) Que hay pocas horas con errores muy grandes que el cuadrado infla
- c) Que el MAE está mal calculado
- d) Que julio tiene pocas horas

**Su respuesta (doble clic para escribir):** ___

**Un caso más.** Finanzas pidió el error "en porcentaje", el **MAPE**: el promedio de |ŷ − y| / y.
Corran la celda y miren el número.

In [ ]:
mape = mean_absolute_percentage_error(mia.departures, mia.recta)
print(f"MAPE de la recta en julio: {mape:,.0%}")          # scikit-learn lo da como fracción; :% lo muestra en porcentaje
print(f"horas de julio con 0 salidas en la estación {estacion}: {(mia.departures == 0).mean():.0%}")

con_salidas = mia[mia.departures > 0]                     # quitando las horas en 0
print(f"MAPE sin las horas en 0: {mean_absolute_percentage_error(con_salidas.departures, con_salidas.recta):,.0%}")

**3. El MAPE salió de más de mil billones por ciento. ¿Qué pasó?**

- a) La recta es pésima en esta estación
- b) Hay horas con 0 salidas y el MAPE divide entre lo real
- c) Hay que dividir entre 100
- d) Es un error de scikit-learn

**Su respuesta (doble clic para escribir):** ___

**4. En las estaciones 32, 50, 196 y 268 la regla tiene R² negativo en julio. ¿Qué significa?**

- a) Que la regla se equivoca hacia abajo
- b) Que la regla es peor que decir siempre el promedio de julio de esa estación
- c) Que hay un error de cálculo: R² va de 0 a 1
- d) Que la estación tiene menos salidas que la semana pasada

**Su respuesta (doble clic para escribir):** ___

## Paso 4. La respuesta (4 min)

Corran la celda: arma el borrador con sus números. Luego escriban su respuesta en la celda de texto de abajo,
en tres renglones, con sus palabras. Escojan la propuesta con menor **MAE a las 8**: es la que mejor contesta la pregunta.

In [ ]:
mejor = tabla["MAE a las 8"].idxmin()                       # la propuesta que menos falla a las 8:00
mae8 = tabla.loc[mejor, "MAE a las 8"]                       # su error promedio a las 8:00, lunes a viernes
hasta = tabla.loc[mejor, "8 de 10 días laborables, error de hasta"]
print(f"1. El número: mañana a las 8:00 esperamos unas {manana[mejor]:.0f} salidas en la estación {estacion} ({mejor}).")
print(f"2. Qué tan seguro: en julio, a las 8:00 de lunes a viernes, fallamos en promedio por {mae8:.1f} bicis, "
      f"y en 8 de cada 10 días laborables por {hasta:.0f} o menos.")
# El pronóstico más el MAE, redondeado: no salen fracciones de bici. Suponemos que a las 8:00
# la estación está llena y que no llegan bicis durante la hora.
plan = round(manana[mejor] + mae8)
anclajes = estaciones.loc[estacion, "capacidad"]             # cuántas bicis caben en la estación
print(f"3. Recomendación y riesgo: planear para unas {plan} salidas (el pronóstico más el MAE); la estación "
      f"tiene {anclajes} anclajes" + (", así que habrá que reabastecerla durante la hora" if plan > anclajes else "")
      + f". En 2 de cada 10 mañanas laborables de julio fallamos por más de {hasta:.0f}.")

**Su respuesta a operaciones** (doble clic para escribir):

1. **El número:** mañana, lunes 3 de agosto, a las 8:00, en la estación ___ esperamos unas ___ salidas.
2. **Qué tan seguro:** en julio, a esa hora y de lunes a viernes, fallamos en promedio por ___ bicis; en 8 de cada 10 días laborables, por ___ o menos.
3. **Recomendación y riesgo:** ___ (para cuántas salidas planear, si caben en los anclajes, y qué pasa en los días en que fallamos más).

Dejen su respuesta escrita en la hoja; al volver, la revisamos juntos.

---
### Antes de entregar
1. *Reiniciar y ejecutar todo*: las autocomprobaciones deben decir ✅.
2. Revisa que tus dos respuestas estén **dentro** de las celdas **Tu respuesta** (no en celdas nuevas).
3. Descarga el notebook (*Archivo*, *Descargar*, *Descargar .ipynb*) y súbelo a Moodle.